In [21]:
import pandas as pd
import numpy as np
import random

In [2]:
df = pd.read_csv(r"C:\Users\Usuario\Desktop\the bridge\github\Desafio_Grupo1\data\synthetic_fin_data_CLEAN.csv")

In [7]:
fraud = df[df["isFraud"] == 1]

print("── Transaction types ──")
print(fraud["type"].value_counts())

print("\n── Countries ──")
print(fraud["ip_country"].value_counts())

print("\n── Merchant categories ──")
print(fraud["merchant_category"].value_counts())

print("\n── Amount stats ──")
print(fraud["amount"].describe())

print("\n── Hour of day stats ──")
print(fraud["hour_of_the_day"].value_counts().sort_index())

── Transaction types ──
type
CASH_OUT    4116
TRANSFER    4097
Name: count, dtype: int64

── Countries ──
ip_country
VE    1000
NG     990
CN     987
CI     959
KH     953
ES     705
DE     701
FR     664
US     632
GB     622
Name: count, dtype: int64

── Merchant categories ──
merchant_category
crypto         2713
electronics    2667
restaurant      584
grocery         574
pharmacy        570
transport       568
fuel            537
Name: count, dtype: int64

── Amount stats ──
count    8.213000e+03
mean     1.467967e+06
std      2.404253e+06
min      0.000000e+00
25%      1.270913e+05
50%      4.414234e+05
75%      1.517771e+06
max      1.000000e+07
Name: amount, dtype: float64

── Hour of day stats ──
hour_of_the_day
0     300
1     358
2     372
3     326
4     274
5     366
6     358
7     328
8     368
9     341
10    375
11    324
12    339
13    346
14    353
15    341
16    345
17    353
18    343
19    342
20    340
21    347
22    351
23    323
Name: count, dtype: int64


In [8]:
# obtenemos los usuarios con historial de transacciones legítimas
legit_users = df[df["isFraud"] == 0].groupby("nameOrig").agg(
    tx_count        = ("amount", "count"),
    avg_amount      = ("amount", "mean"),
    usual_country   = ("ip_country", lambda x: x.mode()[0]),
    usual_merchant  = ("merchant_category", lambda x: x.mode()[0]),
    usual_hour_min  = ("hour_of_the_day", "min"),
    usual_hour_max  = ("hour_of_the_day", "max")
).reset_index()

# filtramos usuarios con suficiente historial para construir un perfil de comportamiento
legit_users = legit_users[legit_users["tx_count"] >= 3]

print(legit_users.shape)
print(legit_users.head())

(3648, 7)
        nameOrig  tx_count     avg_amount usual_country usual_merchant  \
85   C1000610673         3  318007.583333            ES    electronics   
120  C1000940226         3  252720.126667            DE         crypto   
159   C100110245         3   37299.660000            DE    electronics   
242  C1001791426         3  253634.003333            DE    electronics   
443   C100325696         3  169936.186667            ES    electronics   

     usual_hour_min  usual_hour_max  
85               11              19  
120              10              21  
159               0              20  
242              14              16  
443              11              19  


In [9]:
# filtramos usuarios con perfil genuinamente legítimo
# país seguro, categoría segura e importe medio razonable
safe_countries  = ["GB", "FR", "DE", "ES", "US"]
safe_categories = ["grocery", "restaurant", "pharmacy", "fuel", "transport"]

legit_profile_users = legit_users[
    (legit_users["usual_country"].isin(safe_countries)) &
    (legit_users["usual_merchant"].isin(safe_categories)) &
    (legit_users["avg_amount"] < 50000)
].reset_index(drop=True)

print(legit_profile_users.shape)
print(legit_profile_users.head())

(31, 7)
      nameOrig  tx_count    avg_amount usual_country usual_merchant  \
0   C113642551         3   9398.283333            DE        grocery   
1  C1142502638         3  11713.456667            DE      transport   
2   C118451715         3  24915.310000            DE        grocery   
3  C1246965344         3   1527.790000            GB           fuel   
4  C1335728182         3   5748.076667            FR           fuel   

   usual_hour_min  usual_hour_max  
0               9              18  
1              12              18  
2               9              19  
3              10              18  
4              11              18  


In [11]:
print(df["isFraud"].value_counts(normalize=True))
print(df["isFraud"].value_counts())

isFraud
0    0.974305
1    0.025695
Name: proportion, dtype: float64
isFraud
0    311423
1      8213
Name: count, dtype: int64


In [13]:
# objetivo: 6% fraude, 94% legítimas
# mantenemos los 8,213 fraudes obvios existentes
# calculamos cuántas legítimas necesitamos

n_fraud_obvious = 8213

# si fraude total = 6% del total
# y queremos ~50/50 entre fraude obvio y stealth
# entonces fraude stealth ≈ 8,213 también

n_fraud_stealth = 8213  # mismo número que fraude obvio
n_fraud_total = n_fraud_obvious + n_fraud_stealth  # 16,426

# legítimas necesarias para que fraude = 6%
# n_fraud / total = 0.06
# total = n_fraud / 0.06
total_target = n_fraud_total / 0.06
n_legit_target = int(total_target - n_fraud_total)

print(f"Fraude obvio: {n_fraud_obvious}")
print(f"Fraude stealth a generar: {n_fraud_stealth}")
print(f"Fraude total: {n_fraud_total}")
print(f"Legítimas necesarias: {n_legit_target}")
print(f"Total dataset: {int(total_target)}")
print(f"Ratio fraude: {n_fraud_total / total_target * 100:.1f}%")

Fraude obvio: 8213
Fraude stealth a generar: 8213
Fraude total: 16426
Legítimas necesarias: 257340
Total dataset: 273766
Ratio fraude: 6.0%


In [14]:
# instead of requiring ALL THREE conditions
# just require safe country OR safe merchant
# and avg_amount under a reasonable threshold

legit_profile_users = legit_users[
    (legit_users["usual_country"].isin(safe_countries)) |
    (legit_users["usual_merchant"].isin(safe_categories))
].reset_index(drop=True)

print(legit_profile_users.shape)

(3241, 7)


In [15]:
print(df["step"].describe())
print(df[df["isFraud"] == 0]["step"].describe())

count    319636.000000
mean        246.282415
std         145.907995
min           1.000000
25%         156.000000
50%         249.000000
75%         346.000000
max         743.000000
Name: step, dtype: float64
count    311423.000000
mean        243.061505
std         142.169342
min           1.000000
25%         155.000000
50%         238.000000
75%         334.000000
max         718.000000
Name: step, dtype: float64


In [20]:
# cuántos usuarios tienen categoría segura Y país seguro
legit_profile_users_clean = legit_profile_users[
    legit_profile_users["usual_merchant"].isin(safe_categories)
].reset_index(drop=True)

print(legit_profile_users_clean.shape)
print(legit_profile_users_clean["usual_merchant"].value_counts())
print(legit_profile_users_clean["usual_country"].value_counts())

(673, 7)
usual_merchant
fuel          203
grocery       141
pharmacy      122
restaurant    108
transport      99
Name: count, dtype: int64
usual_country
CI    182
CN    142
DE     95
ES     77
FR     54
GB     52
KH     23
NG     20
VE     17
US     11
Name: count, dtype: int64


In [19]:
# filtramos usuarios con categoría de comercio segura Y país seguro
legit_profile_users_clean = legit_profile_users[
    (legit_profile_users["usual_merchant"].isin(safe_categories)) &
    (legit_profile_users["usual_country"].isin(safe_countries))
].reset_index(drop=True)

# verificamos cuántos usuarios quedan y su distribución
print(legit_profile_users_clean.shape)
print(legit_profile_users_clean["usual_merchant"].value_counts())
print(legit_profile_users_clean["usual_country"].value_counts())

(289, 7)
usual_merchant
fuel          74
restaurant    57
transport     55
pharmacy      54
grocery       49
Name: count, dtype: int64
usual_country
DE    95
ES    77
FR    54
GB    52
US    11
Name: count, dtype: int64


In [23]:
# verificamos qué países tiene legit_profile_users_clean
print(legit_profile_users_clean["usual_country"].value_counts())

usual_country
CI    182
CN    142
DE     95
ES     77
FR     54
GB     52
KH     23
NG     20
VE     17
US     11
Name: count, dtype: int64


In [24]:
# re-aplicamos el filtro correctamente
legit_profile_users_clean = legit_profile_users[
    (legit_profile_users["usual_merchant"].isin(safe_categories)) &
    (legit_profile_users["usual_country"].isin(safe_countries))
].reset_index(drop=True)

print(legit_profile_users_clean.shape)
print(legit_profile_users_clean["usual_country"].value_counts())

(289, 7)
usual_country
DE    95
ES    77
FR    54
GB    52
US    11
Name: count, dtype: int64


In [25]:
random.seed(42)
np.random.seed(42)

legit = df[df["isFraud"] == 0]
stealth_rows = []

for _, user in legit_profile_users_clean.iterrows():

    # ── señal 4: mismo destino para todas las transacciones de este usuario ──
    shared_dest = legit["nameDest"].sample(1).values[0]

    # ── señal 1: steps muy cercanos — ventana de 27-30 steps ────────────────
    base_step = random.randint(1, 700)
    n_txs = random.randint(27, 30)
    steps = [base_step + i for i in range(n_txs)]

    for step in steps:

        # ── señal 2: amount = 60-80% del saldo — capeado a €500 ─────────────
        oldbalanceOrg  = round(random.uniform(500, 5000), 2)
        amount         = round(oldbalanceOrg * random.uniform(0.60, 0.80), 2)
        amount         = max(10.0, min(amount, 500.0))
        newbalanceOrig = round(max(0.0, oldbalanceOrg - amount), 2)

        oldbalanceDest = round(random.uniform(1000, 20000), 2)
        newbalanceDest = round(oldbalanceDest + amount, 2)

        # ── señal 3: hora nocturna suave ─────────────────────────────────────
        hour = random.randint(22, 23) if random.random() > 0.5 else random.randint(0, 2)

        stealth_rows.append({
            "step":                step,
            "type":                random.choice(safe_types),
            "amount":              amount,
            "nameOrig":            user["nameOrig"],
            "oldbalanceOrg":       oldbalanceOrg,
            "newbalanceOrig":      newbalanceOrig,
            "nameDest":            shared_dest,
            "oldbalanceDest":      oldbalanceDest,
            "newbalanceDest":      newbalanceDest,
            "isFraud":             1,
            "hour_of_the_day":     hour,
            "ip_country":          user["usual_country"],
            "merchant_category":   user["usual_merchant"],
            "balance_discrepancy": oldbalanceOrg - amount - newbalanceOrig
        })

df_stealth = pd.DataFrame(stealth_rows)

print(f"Stealth fraud rows generadas: {len(df_stealth)}")
print(df_stealth[["amount", "ip_country", "merchant_category", "hour_of_the_day"]].head(10))

Stealth fraud rows generadas: 8220
   amount ip_country merchant_category  hour_of_the_day
0  401.22         DE         transport               22
1  397.86         DE         transport               22
2  500.00         DE         transport                0
3  500.00         DE         transport                0
4  500.00         DE         transport               23
5  500.00         DE         transport               23
6  500.00         DE         transport                0
7  500.00         DE         transport               22
8  500.00         DE         transport               22
9  500.00         DE         transport                2


In [28]:
# fraude total = 8213 obvio + 8220 stealth
n_fraud_total = len(df[df["isFraud"] == 1]) + len(df_stealth)

# legítimas necesarias para 6% fraude
# n_fraud / total = 0.06 → total = n_fraud / 0.06
total_target  = n_fraud_total / 0.06
n_legit_target = int(total_target - n_fraud_total)

n_legit_current = len(df[df["isFraud"] == 0])
n_legit_to_drop = n_legit_current - n_legit_target

print(f"Fraude total: {n_fraud_total}")
print(f"Legítimas actuales: {n_legit_current}")
print(f"Legítimas necesarias: {n_legit_target}")
print(f"Legítimas a eliminar: {n_legit_to_drop}")

Fraude total: 16433
Legítimas actuales: 311423
Legítimas necesarias: 257450
Legítimas a eliminar: 53973


In [29]:
# eliminamos 53,973 filas legítimas aleatorias
legit_trimmed = df[df["isFraud"] == 0].sample(
    n=257450, random_state=42
).reset_index(drop=True)

# combinamos legítimas + fraude obvio + fraude stealth
df_round2 = pd.concat([
    legit_trimmed,
    df[df["isFraud"] == 1],
    df_stealth
]).reset_index(drop=True)

print(f"Total rows: {len(df_round2)}")
print(df_round2["isFraud"].value_counts())
print(df_round2["isFraud"].value_counts(normalize=True))

Total rows: 273883
isFraud
0    257450
1     16433
Name: count, dtype: int64
isFraud
0    0.94
1    0.06
Name: proportion, dtype: float64


In [30]:
df_round2.to_csv(
    r"C:\Users\Usuario\Desktop\the bridge\github\Desafio_Grupo1\data\synthetic_fin_data_ROUND2.csv",
    index=False
)

print("Exported successfully!")

Exported successfully!
